# Freeze the parking V0/V1 depth benchmark — CPU only

**PARKING-V0-V1-BENCHMARK-CPU-2026-10-28-r1**

Use **Runtime → Change runtime type → Hardware accelerator: None**. This notebook reads the reviewed complete nuScenes 0–6 m coverage run already on Drive. It freezes identical RGB inputs and sparse LiDAR masks for the unchanged V0 baseline and a later trained V1 candidate. It does **not** load models, run inference, train, download new datasets, or update the phone.

The package has separate original-role **dev** and **heldout_eval** shards. The three previously inspected evaluation logs are excluded; train and previously excluded Mini logs are never exported. Original split assignments stay unchanged. Selection uses GT support only—not model predictions.

## What is being frozen?

- Up to **512 images per role**, with four workers and **32-image immutable shards** on Drive.
- Every eligible Pedestrian 0–2 m view and Vehicle/Pedestrian 0–1 m view is retained first. Remaining places use a deterministic class/range round robin with recording-log fairness and unused keyframes. Mandatory rare views can share a keyframe; they are not independent samples.
- Original RGB, original pixel coordinates, camera intrinsics and conservative, unambiguous, exact-semantic object surface points. Each scored object/camera/band requires at least five measured points; GT holes remain holes.
- All six bands: **0–1, 1–2, 0–2, 2–4, 4–6, 0–6 m**. Distance is **camera-axis Z in metres**, not Euclidean camera range or ultrasonic nearest-surface range.

**Primary score:** equal Vehicle/Pedestrian × 2–4/4–6 m cells. Within a cell, average cameras within an object/keyframe observation, then observations within a track, tracks within a recording log, then logs. This keeps large cars and repeated video frames from dominating. Also report pointwise mean/median/p90 absolute error and prediction coverage. Errors are conditional on valid positive finite predictions, so coverage must always accompany them; V0/V1 also get a paired common-valid-mask comparison.

0–2 m is reported separately; 0–1 m is exploratory because support is sparse. **No ±10 cm pass/fail requirement.** These are GT-oracle-class depth metrics, not detection mAP, 3D cuboid accuracy or product qualification. V0/V1 model and preprocessing hashes must be registered before the later inference step; this notebook intentionally does not choose weights.

## 1. Mount Drive and create a separate run

Keep the same RUN_ID to resume the same code/protocol. Never run two notebooks against the same output simultaneously. Do not update code midway through a run. Completed historical coverage files are read-only.

Source archives and results stay on Drive; temporary metadata, labels and only the active archive live on Colab disk. Your Mac is not used.

In [ ]:
from google.colab import drive
from pathlib import Path
from collections import deque
import datetime, json, os, shlex, subprocess, sys

REVISION = 'PARKING-V0-V1-BENCHMARK-CPU-2026-10-28-r1'
print(REVISION, flush=True)
if os.environ.get('COLAB_GPU', '').lower() not in ('', '0', 'none'):
    raise RuntimeError('Choose Hardware accelerator: None; this preparation is CPU-only.')
drive.mount('/content/drive')
RUN_ID = 'parking_v0_v1_frozen_benchmark_r1'
if not RUN_ID or any(ch not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for ch in RUN_ID):
    raise ValueError('Use letters, digits, underscore and hyphen in RUN_ID.')
MOBILE = Path('/content/mobile_adas3d')
DATASET = Path('/content/drive/MyDrive/mobile_adas3d_datasets/nuscenes_v1_0')
COVERAGE = Path('/content/drive/MyDrive/mobile_adas3d_outputs/parking_near_field_coverage/nuscenes_full_0_6m_r1')
OUTPUT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/parking_v0_v1_benchmark') / RUN_ID
LOCAL = Path('/content') / ('parking_benchmark_' + RUN_ID)
DEPS = Path('/content/parking_benchmark_cpu_deps')
PYTHON = Path(sys.executable)
WORKERS = 4
READY = False
OUTPUT.mkdir(parents=True, exist_ok=True)

def run(command, name, cwd=None, env=None):
    command = [str(item) for item in command]
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    log = OUTPUT/'colab_logs'/f'{name}_{stamp}.log'; log.parent.mkdir(parents=True, exist_ok=True)
    merged = os.environ.copy(); merged.update(env or {})
    print('+', shlex.join(command), '\nDurable log:', log, flush=True)
    tail = deque(maxlen=30)
    with log.open('w') as stream:
        process = subprocess.Popen(command, cwd=cwd, env=merged, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True); stream.write(line); stream.flush(); tail.append(line.rstrip())
        finally:
            process.stdout.close()
        status = process.wait(); stream.write(f'\nEXIT CODE: {status}\n'); stream.flush()
    if status:
        raise RuntimeError(f'Exit {status}; preserve the run and inspect {log}\n'+'\n'.join(tail))
    return log

print('Read-only coverage:', COVERAGE, '\nNew benchmark:', OUTPUT, '\nLocal scratch:', LOCAL)

## 2. Pull main, install only pinned CPU task packages, and test

No old M62/M67 runtime, PyTorch, CUDA compiler, Core ML or nuScenes SDK is needed. Four packages are installed in a separate task import directory. The global Colab environment is not repinned. A dirty tracked checkout or unexpected branch is preserved, not reset.

In [ ]:
url = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not MOBILE.exists():
    run(['git', 'clone', '--branch', 'main', url, MOBILE], 'clone')
branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=MOBILE, text=True).strip()
if branch != 'main':
    raise RuntimeError(f'Expected main, found {branch!r}. Preserve it and use a clean main checkout.')
changes = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=MOBILE, text=True)
if changes.strip():
    raise RuntimeError('Tracked changes exist. Preserve them and use a clean checkout; no reset is performed.')
run(['git', 'pull', '--ff-only'], 'update', MOBILE)
SCRIPT = MOBILE/'scripts/prepare_parking_v0_v1_benchmark.py'
CONFIG = MOBILE/'configs/parking_v0_v1_benchmark_v1.json'
for path in [SCRIPT, CONFIG, MOBILE/'scripts/parking_v0_v1_metrics.py',
             MOBILE/'tests/test_parking_v0_v1_benchmark.py', MOBILE/'tests/test_nuscenes_full_coverage.py',
             COVERAGE/'nuscenes_0_6m_coverage_results.zip', DATASET/'download_manifest_keyframes_all.json']:
    if not path.is_file():
        raise FileNotFoundError(f'Missing required input: {path}. Correct the path before preparation.')
protocol = json.loads(CONFIG.read_text())
if protocol['revision'] != REVISION:
    raise RuntimeError('Notebook/protocol revision differs. Reload the matching notebook before running.')
cov_config = json.loads((MOBILE/'configs/nuscenes_coverage_0_6m_v1.json').read_text())
requirements = [name+'=='+version for name, version in cov_config['runtime_packages'].items()]
run([PYTHON, '-m', 'pip', 'install', '--quiet', '--upgrade', '--only-binary=:all:', '--no-deps',
     '--target', DEPS, *requirements], 'cpu_dependencies', MOBILE)
ENV = {'PYTHONPATH': os.pathsep.join([str(DEPS), str(MOBILE)]), 'CUDA_VISIBLE_DEVICES': '',
       'OMP_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1',
       'PYTHONDONTWRITEBYTECODE': '1'}
for pattern in ['test_nuscenes_full_coverage.py', 'test_parking_v0_v1_benchmark.py']:
    run([PYTHON, '-m', 'unittest', 'discover', '-s', 'tests', '-p', pattern, '-v'],
        'tests_'+pattern.replace('.py',''), MOBILE, ENV)

def stage(name):
    return run([PYTHON, '-X', 'faulthandler', '-u', SCRIPT, name, '--dataset', DATASET,
                '--coverage', COVERAGE, '--output', OUTPUT, '--local', LOCAL, '--workers', str(WORKERS)],
               'benchmark_'+name, MOBILE, ENV)

READY = True
print('CPU task ready. No sensor archive or model has been processed.')

## 3. Freeze the GT-only selection first

Verifies the reviewed coverage ZIP/identities, all 537 compressed chunks and the unchanged source/split map. This reads small coverage records, not the 45 GB sensor data. Selected dev and held-out image IDs are frozen before any model inference. The inventory reports actual unique tracks and logs; the 512-image budget is a cap, not a promise of independent samples.

In [ ]:
if not globals().get('READY', False):
    raise RuntimeError('Run sections 1 and 2 first.')
stage('plan')
selection = json.loads((OUTPUT/'selection_manifest.json').read_text())
for role, result in selection['selected_pool'].items():
    print('\n', role, ':', result['images'], 'images;', result['unique_keyframes'], 'keyframes;', result['recording_logs'], 'logs')
    for cell in protocol['selection_cells']:
        row = result['cells'][cell]
        print(cell, '| observations:', row['annotation_observations'], '| tracks:', row['instance_tracks'],
              '| logs:', row['recording_logs'], '| reference points:', row['eligible_reference_points'])

## 4. Prepare original RGB and frozen sparse GT shards

The notebook stages one source archive at a time, verifies its SHA256, and extracts only the selected RGB files and referenced LiDAR scans locally. Each selected reprojection must reproduce the reviewed coverage counts. It then writes separate dev/held-out shards with RGB, sparse arrays and exact cell masks. A minimum of five conservative, exact-semantic, unambiguous points per object/camera/band is required for that cell.

This may take time because each required archive is read sequentially from Drive. **Keep the GPU off.** Progress and logs are saved. Local disk checks leave an 8 GB reserve. Completed part scratch is cleaned only after its shards and receipt are durably verified; Drive source archives and historical runs are never deleted.

**Resuming:** if a cell stops, stop the old process before rerunning it. If the runtime resets, rerun from the top with the same RUN_ID. Completed 32-image shards are verified and skipped; only an unfinished shard or active archive staging step may repeat.

In [ ]:
if not globals().get('READY', False):
    raise RuntimeError('Run sections 1 and 2 first.')
stage('prepare')
report = json.loads((OUTPUT/'preparation_report.json').read_text())
assert report['complete'] and not report['model_artifacts_bound']
assert not report['model_inference_performed'] and not report['training_performed']
assert not report['accuracy_pass_fail_enabled'] and report['accuracy_threshold_m'] is None
print('Prepared:', report['images'], 'images in', report['shards'], 'shards; GB:', round(report['shard_bytes']/1e9, 3))
print('No V0/V1 scores yet. Model artifact identities must be registered before later inference.')

## 5. Download the small preparation report

Return **parking_v0_v1_benchmark_results.zip** for review. It includes the frozen selection, sample/shard manifests, metric contract, support counts and an **unbound** V0/V1 registration template. RGB/GT shards stay on Drive and are not duplicated in this small report ZIP.

Next—not performed here: register the exact unchanged V0 artifact plus preprocessing/output-sampling identities, then measure it. Later train an authorized V1 candidate on training-only data and compare it on these identical frozen inputs. Do not tune using held-out scores or introduce per-image scale fitting.

In [ ]:
stage('bundle')
BUNDLE = OUTPUT/'parking_v0_v1_benchmark_results.zip'
print('Saved:', BUNDLE)
from google.colab import files
files.download(str(BUNDLE))